# LeetCode 449 — Serialize and Deserialize BST bằng Python

Link đề: https://leetcode.com/problems/serialize-and-deserialize-bst/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Search Tree, Design, DFS, BFS, String

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None

class Codec:
    def serialize(self, root: Optional[TreeNode]) -> str:
        """Encodes a tree to a single string."""

    def deserialize(self, data: str) -> Optional[TreeNode]:
        """Decodes your encoded data to tree."""

# Your Codec object will be instantiated and called as such:
# ser = Codec()
# deser = Codec()
# tree = ser.serialize(root)
# ans = deser.deserialize(tree)
# return ans
```

Ghi chú: đề được viết lại theo hiểu biết về đề gốc (đề gốc của bài này khá ngắn, không có phần kiểm tra "độ gọn" cụ thể). Nên đối chiếu lại đề gốc nếu muốn khớp từng chữ. Notebook tự dựng `TreeNode`, các hàm dựng/so sánh cây và một bộ chạy mô phỏng cách LeetCode gọi lớp `Codec` (một đối tượng để `serialize`, một đối tượng **khác** để `deserialize`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế** (design): viết lớp `Codec` có hai phương thức `serialize` (cây → chuỗi) và `deserialize` (chuỗi → cây) sao cho **đi một vòng khứ hồi (round-trip) ra đúng cây ban đầu**, và chuỗi **càng gọn càng tốt**.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, duyệt cây DFS (tiền tự/preorder), `str.join` / `str.split` / `map`, hàm lồng nhau với `nonlocal`, ngăn xếp (`list`) dùng như stack, `chr`/`ord` để nén số vào ký tự.
3. Đi từ **cách tổng quát (ghi cả `#` cho nút rỗng)** → **chỉ ghi tiền tự + giới hạn `[min, max]`** → **chèn lần lượt vào BST** → **dùng stack không đệ quy** → **nén mỗi số thành một ký tự**.
4. Hiểu mấu chốt: BST có thứ tự trung tự (inorder) **đã được sắp xếp** sẵn, nên chỉ cần **một** trong các thứ tự duyệt (tiền tự) là đủ khôi phục cây — **không cần ghi nút rỗng**; mỗi giá trị chỉ "được phép" đứng ở vị trí nằm trong khoảng `(min, max)` do tổ tiên quy định.
5. Tránh các lỗi kinh điển: lưu trạng thái trong đối tượng (vì `ser` và `deser` là hai đối tượng khác nhau), quên xử lý cây rỗng / chuỗi rỗng, nhầm cận trên/cận dưới của hai con, và quên rằng cây lệch 10^4 nút có thể làm đệ quy Python tràn stack.

## 2. Đề bài & ví dụ

**Serialization** (tuần tự hoá) là biến một cấu trúc dữ liệu thành một chuỗi (hoặc dãy bit) để lưu xuống file, bộ nhớ, hoặc gửi qua mạng; sau đó có thể dựng lại cấu trúc ban đầu ở nơi khác (**deserialization**).

Hãy thiết kế thuật toán tuần tự hoá / giải tuần tự hoá một **cây tìm kiếm nhị phân (BST)**. Không giới hạn cách làm, miễn là BST được mã hoá thành một chuỗi và chuỗi đó giải mã lại **ra đúng cấu trúc cây ban đầu**.

**Chuỗi mã hoá phải càng gọn càng tốt.**

**Ví dụ 1**

- Input: `root = [2,1,3]`
- Output: `[2,1,3]` (cây sau khi `serialize` rồi `deserialize` phải y hệt)

**Ví dụ 2**

- Input: `root = []`
- Output: `[]`

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[0, 10^4]`.
- `0 <= Node.val <= 10^4`
- Cây đầu vào được **đảm bảo** là một BST.

Chú ý quan trọng khi LeetCode chấm: nó tạo **hai đối tượng `Codec` khác nhau** (`ser` và `deser`); nên bạn **không được** nhét thông tin vào thuộc tính của đối tượng ở bước `serialize` rồi mong bước `deserialize` đọc lại. Mọi thứ cần thiết phải nằm trong **chuỗi**.

Trong notebook này ta giả sử các giá trị trong BST **đôi một khác nhau** (BST chuẩn). Nếu có giá trị trùng, cấu trúc cây từ tiền tự sẽ **mơ hồ** — xem mục 15.

In [1]:
from typing import Optional, List
from collections import deque, Counter
import random
import sys
import time


class TreeNode:
    def __init__(self, x=0, left=None, right=None):
        self.val = x
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def same_tree(a, b):
    """So sánh hai cây có CÙNG HÌNH DẠNG và CÙNG GIÁ TRỊ (không dùng đệ quy để chịu được cây sâu)."""
    stack = [(a, b)]
    while stack:
        x, y = stack.pop()
        if x is None and y is None:
            continue
        if x is None or y is None or x.val != y.val:
            return False
        stack.append((x.left, y.left))
        stack.append((x.right, y.right))
    return True


examples = [[2, 1, 3], []]
for vals in examples:
    t = build_tree(vals)
    assert tree_to_list(t) == vals
    assert same_tree(t, build_tree(vals))
    print(vals, "-> dựng cây rồi đọc lại:", tree_to_list(t))

[2, 1, 3] -> dựng cây rồi đọc lại: [2, 1, 3]
[] -> dựng cây rồi đọc lại: []


## 3. Kiến thức nền

### 3.1. Cây tìm kiếm nhị phân (BST) và thứ tự trung tự

Một BST có tính chất: với **mọi** nút `x`, **mọi** giá trị trong cây con trái đều `< x.val` và **mọi** giá trị trong cây con phải đều `> x.val`. Hệ quả rất đẹp: duyệt **trung tự (inorder: trái → nút → phải)** cho ra dãy **tăng dần**.

In [2]:
def inorder(node):
    return inorder(node.left) + [node.val] + inorder(node.right) if node else []


t = build_tree([5, 3, 6, 2, 4, None, 7])
print(inorder(t))                         # [2, 3, 4, 5, 6, 7]  <- đã sắp xếp
assert inorder(t) == sorted(inorder(t))

[2, 3, 4, 5, 6, 7]


### 3.2. Duyệt tiền tự (preorder: nút → trái → phải)

Duyệt tiền tự ghi **nút cha trước các con**. Nhờ vậy phần tử **đầu tiên** của dãy luôn là **gốc**. Ta sẽ dùng chính tính chất này để khôi phục cây.

In [3]:
def preorder(node):
    return [node.val] + preorder(node.left) + preorder(node.right) if node else []


t = build_tree([5, 3, 6, 2, 4, None, 7])
print(preorder(t))                         # [5, 3, 2, 4, 6, 7]

[5, 3, 2, 4, 6, 7]


### 3.3. `join`, `split`, `map` — biến list số thành chuỗi và ngược lại

`" ".join(list_of_str)` ghép các chuỗi, cách nhau một dấu cách. `s.split()` cắt chuỗi theo khoảng trắng. `map(int, ...)` đổi từng phần tử sang số nguyên. Chú ý: `"".split()` trả về **list rỗng** (không phải `[""]`) — điều này tiện cho cây rỗng.

In [4]:
nums = [5, 3, 2, 4, 6, 7]
s = " ".join(map(str, nums))
print(repr(s))                              # '5 3 2 4 6 7'
print(list(map(int, s.split())))            # [5, 3, 2, 4, 6, 7]
print("".split(), "".split() == [])         # [] True

'5 3 2 4 6 7'
[5, 3, 2, 4, 6, 7]
[] True


### 3.4. Cận vô cực `float("-inf")` / `float("inf")`

Để bắt đầu với khoảng "chưa có giới hạn", ta dùng `-inf` và `inf`: mọi số đều nằm giữa chúng.

In [5]:
inf = float("inf")
print(-inf < 0 < inf, -inf < -10**9 < inf)    # True True
print(0 <= 5 <= 10, 0 <= 11 <= 10)           # True False   <- chuỗi so sánh a <= x <= b

True True
True False


### 3.5. Hàm lồng nhau + `nonlocal` (con trỏ chạy dọc dãy)

Khi giải mã, ta đọc dãy số **từ trái sang phải đúng một lần**; cần một "con trỏ" `i` dùng chung cho mọi lần gọi đệ quy. Khai báo `nonlocal i` để hàm con **gán lại** `i` của hàm ngoài (không khai báo thì Python tạo biến cục bộ mới và báo `UnboundLocalError`).

In [6]:
def demo():
    i = 0
    def advance():
        nonlocal i
        i += 1
    advance(); advance()
    return i


print(demo())      # 2

2


### 3.6. Stack bằng `list` và cách LeetCode gọi `Codec`

Stack (ngăn xếp): vào sau ra trước. Với `list`: `append` để đẩy, `pop()` để lấy ra, `stack[-1]` để xem đỉnh. Ta cũng dựng **bộ chạy mô phỏng LeetCode**: tạo hai đối tượng `Codec` khác nhau, `serialize` bằng đối tượng 1, `deserialize` bằng đối tượng 2.

In [7]:
def run_codec(codec_cls, root):
    """Mô phỏng cách LeetCode gọi: ser và deser là HAI đối tượng khác nhau."""
    ser = codec_cls()
    deser = codec_cls()
    data = ser.serialize(root)
    assert isinstance(data, str), "serialize phải trả về str"
    ans = deser.deserialize(data)
    return data, ans


stack = []
stack.append(1); stack.append(2); stack.append(3)
print(stack[-1], stack.pop(), stack)         # 3 3 [1, 2]

3 3 [1, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét BST `[5,3,6,2,4,null,7]`:

```
        5
       / \
      3   6
     / \   \
    2   4   7
```

Duyệt tiền tự cho `5 3 2 4 6 7`. Thử **tự khôi phục** cây chỉ từ dãy này (không có dấu `#`):

1. Số đầu tiên `5` là **gốc**.
2. Các số kế tiếp `3 2 4` đều **nhỏ hơn 5** → chúng thuộc cây con **trái**. Số `3` đứng đầu nhóm này nên là gốc cây con trái.
3. Trong `2 4` (sau `3`): `2 < 3` → cây con trái của `3`; `4 > 3` nhưng `4 < 5` → cây con phải của `3`.
4. Các số còn lại `6 7` đều **lớn hơn 5** → cây con phải; `6` đứng đầu là gốc, `7 > 6` là con phải của `6`.

Điểm mấu chốt: **ta chưa hề cần dấu "rỗng"**. Chỉ cần so sánh số với gốc là biết nó thuộc bên trái hay bên phải, và nhóm "nhỏ hơn gốc" luôn đứng liền nhau ngay sau gốc trong tiền tự. Hãy kiểm chứng bằng code nhận xét "nhóm nhỏ hơn đứng liền sau gốc".

In [8]:
pre = preorder(build_tree([5, 3, 6, 2, 4, None, 7]))
root_val = pre[0]
smaller = [x for x in pre[1:] if x < root_val]
bigger = [x for x in pre[1:] if x > root_val]
print("tiền tự     :", pre)
print("nhỏ hơn gốc :", smaller)
print("lớn hơn gốc :", bigger)
assert pre[1:] == smaller + bigger          # nhóm nhỏ đứng trước, liền nhau, rồi tới nhóm lớn

tiền tự     : [5, 3, 2, 4, 6, 7]
nhỏ hơn gốc : [3, 2, 4]
lớn hơn gốc : [6, 7]


## 5. Cách 1 — Brute Force / baseline: ghi cả nút rỗng bằng `#` (cách cho cây nhị phân bất kỳ)

**Ý tưởng.** Chưa dùng tính chất BST. Duyệt tiền tự, gặp nút rỗng thì ghi `#`. Ví dụ cây `[2,1,3]` → `"2 1 # # 3 # #"`. Khi giải mã, đọc từng token: `#` → `None`; số → tạo nút, đọc tiếp để dựng con trái, rồi con phải (cùng thứ tự đã ghi). Cách này đúng cho **mọi cây nhị phân** (bài "Serialize and Deserialize Binary Tree").

**Nút thắt.** Cây `n` nút phải ghi thêm `n + 1` dấu `#`: chuỗi dài gần gấp đôi, và **lãng phí** vì BST đã có thông tin thứ tự rồi — đề yêu cầu "càng gọn càng tốt".

In [9]:
class CodecNull:
    """Cách 1: tiền tự có # cho nút rỗng (tổng quát, không dùng tính chất BST)."""

    def serialize(self, root):
        out = []

        def dfs(node):
            if node is None:
                out.append("#")
                return
            out.append(str(node.val))
            dfs(node.left)
            dfs(node.right)

        dfs(root)
        return " ".join(out)

    def deserialize(self, data):
        tokens = iter(data.split())

        def dfs():
            t = next(tokens)
            if t == "#":
                return None
            node = TreeNode(int(t))
            node.left = dfs()
            node.right = dfs()
            return node

        return dfs()


for vals in ([2, 1, 3], [], [5, 3, 6, 2, 4, None, 7]):
    data, back = run_codec(CodecNull, build_tree(vals))
    assert same_tree(build_tree(vals), back)
    print(vals, "->", repr(data), f"(dài {len(data)} ký tự)")

[2, 1, 3] -> '2 1 # # 3 # #' (dài 13 ký tự)
[] -> '#' (dài 1 ký tự)
[5, 3, 6, 2, 4, None, 7] -> '5 3 2 # # 4 # # 6 # 7 # #' (dài 25 ký tự)


**Độ phức tạp.** Thời gian `O(n)` (mỗi nút và mỗi `#` đúng một lần), bộ nhớ `O(n)` cho chuỗi và `O(h)` cho đệ quy (`h` = chiều cao cây). Đúng nhưng **chuỗi chưa gọn**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** với BST, **thứ tự tiền tự một mình đã xác định duy nhất cây** — không cần `#`.

Vì sao? Phần tử đầu tiên là gốc. Trong BST, tại mỗi vị trí cây ta biết **khoảng giá trị được phép** `(lo, hi)`:

- Gốc: khoảng `(-inf, +inf)`.
- Con trái của nút `x` trong khoảng `(lo, x)`; con phải trong khoảng `(x, hi)`.

Khi đọc giá trị kế tiếp `v` trong dãy tiền tự và đang cần dựng một cây con có khoảng `(lo, hi)`:

- Nếu `lo < v < hi` → `v` chính là gốc của cây con này (vì tiền tự đặt gốc trước), ta tiêu thụ `v`, rồi dựng con trái với `(lo, v)`, con phải với `(v, hi)`.
- Nếu `v` **nằm ngoài khoảng** → cây con này **rỗng**, **không tiêu thụ** `v` (nó thuộc về một cây con khác, ở phía trên).

Không cần `#` vì "rỗng" được suy ra từ việc giá trị kế tiếp **không vừa khoảng**. Hãy xem quyết định đó bằng một hàm nhỏ.

In [10]:
def fits(v, lo, hi):
    return lo < v < hi


inf = float("inf")
# cây gốc 5: con trái cần khoảng (-inf, 5), con phải cần khoảng (5, inf)
print(fits(3, -inf, 5), fits(6, -inf, 5))      # True False   (3 làm con trái, 6 thì không)
print(fits(6, 5, inf), fits(3, 5, inf))        # True False

True False
True False


## 7. Các cách tiếp theo

### Cách 2 — Tiền tự + giới hạn `(lo, hi)` bằng đệ quy (cách "chuẩn")

**Giải thích từng đoạn.**

1. `serialize`: duyệt tiền tự, gom giá trị vào list, `" ".join(map(str, nums))`.
2. `deserialize`: tách chuỗi thành `nums` (list số nguyên), `i = 0` là con trỏ.
3. `dfs(lo, hi)`: nếu hết dãy (`i == len(nums)`) hoặc `nums[i]` ngoài khoảng → trả `None`. Ngược lại tạo nút, `i += 1`, dựng `left = dfs(lo, x)` rồi `right = dfs(x, hi)` — **thứ tự này bắt buộc** (trái trước, đúng như tiền tự).
4. Gọi `dfs(-inf, inf)`.

Code demo rồi code nộp (ở đây giống nhau).

In [11]:
class CodecBounds:
    """Cách 2: tiền tự + giới hạn (lo, hi), đệ quy."""

    def serialize(self, root):
        nums = []

        def dfs(node):
            if node is None:
                return
            nums.append(node.val)
            dfs(node.left)
            dfs(node.right)

        dfs(root)
        return " ".join(map(str, nums))

    def deserialize(self, data):
        nums = list(map(int, data.split()))
        i = 0

        def dfs(lo, hi):
            nonlocal i
            if i == len(nums) or not (lo < nums[i] < hi):
                return None
            x = nums[i]
            i += 1
            node = TreeNode(x)
            node.left = dfs(lo, x)
            node.right = dfs(x, hi)
            return node

        return dfs(float("-inf"), float("inf"))


for vals in ([2, 1, 3], [], [5, 3, 6, 2, 4, None, 7]):
    data, back = run_codec(CodecBounds, build_tree(vals))
    assert same_tree(build_tree(vals), back)
    print(vals, "->", repr(data), f"(dài {len(data)})")

[2, 1, 3] -> '2 1 3' (dài 5)
[] -> '' (dài 0)
[5, 3, 6, 2, 4, None, 7] -> '5 3 2 4 6 7' (dài 11)


**Code nộp LeetCode (Cách 2, dùng khi cây không quá lệch):**

```python
class Codec:
    def serialize(self, root: Optional[TreeNode]) -> str:
        nums = []
        def dfs(node):
            if node is None:
                return
            nums.append(node.val)
            dfs(node.left)
            dfs(node.right)
        dfs(root)
        return " ".join(map(str, nums))

    def deserialize(self, data: str) -> Optional[TreeNode]:
        nums = list(map(int, data.split()))
        i = 0
        def dfs(lo, hi):
            nonlocal i
            if i == len(nums) or not (lo < nums[i] < hi):
                return None
            x = nums[i]
            i += 1
            node = TreeNode(x)
            node.left = dfs(lo, x)
            node.right = dfs(x, hi)
            return node
        return dfs(float("-inf"), float("inf"))
```

**Độ phức tạp.** Mỗi giá trị được tiêu thụ đúng một lần và mỗi lời gọi `dfs` hoặc tạo nút hoặc trả `None` ngay: tổng số lời gọi `≤ 2n + 1` → thời gian `O(n)`; bộ nhớ `O(n)` cho chuỗi/list và `O(h)` cho ngăn xếp đệ quy (cây lệch: `h = n = 10^4`, **có thể tràn đệ quy** Python — xem Cách 4).

### Cách 3 — Tiền tự rồi **chèn lần lượt** vào BST (insert)

**Ý tưởng khác hẳn.** Không cần khái niệm khoảng. Nhận xét: nếu ta **chèn các giá trị theo đúng thứ tự tiền tự** vào một BST ban đầu rỗng bằng thao tác chèn thông thường, cây thu được **y hệt** cây gốc. Lý do: trong tiền tự, mỗi nút xuất hiện **trước** con cháu của nó, nên khi chèn một giá trị, cha của nó chắc chắn đã có sẵn và đường đi tìm chỗ chèn sẽ dẫn đúng tới vị trí cũ.

**Giải thích từng đoạn.** `serialize` giống Cách 2. `deserialize`: `root = None`; với mỗi `x` trong `nums`: nếu `root` rỗng thì `root = TreeNode(x)`; nếu không, đi từ gốc: `x < cur.val` thì sang trái, ngược lại sang phải, đến khi gặp chỗ trống thì gắn nút mới.

In [12]:
class CodecInsert:
    """Cách 3: tiền tự + chèn từng giá trị vào BST (vòng lặp, không đệ quy)."""

    def serialize(self, root):
        out, stack = [], [root] if root else []
        while stack:
            node = stack.pop()
            out.append(node.val)
            if node.right:                 # đẩy phải trước để trái được lấy ra trước
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
        return " ".join(map(str, out))

    def deserialize(self, data):
        root = None
        for x in map(int, data.split()):
            if root is None:
                root = TreeNode(x)
                continue
            cur = root
            while True:
                if x < cur.val:
                    if cur.left is None:
                        cur.left = TreeNode(x)
                        break
                    cur = cur.left
                else:
                    if cur.right is None:
                        cur.right = TreeNode(x)
                        break
                    cur = cur.right
        return root


for vals in ([2, 1, 3], [], [5, 3, 6, 2, 4, None, 7]):
    data, back = run_codec(CodecInsert, build_tree(vals))
    assert same_tree(build_tree(vals), back)
    print(vals, "->", repr(data))

[2, 1, 3] -> '2 1 3'
[] -> ''
[5, 3, 6, 2, 4, None, 7] -> '5 3 2 4 6 7'


**Độ phức tạp.** Mỗi lần chèn tốn `O(độ sâu)`. Cây cân bằng: `O(n log n)`; cây lệch (chuỗi): `O(n²)` (với `n = 10^4` khoảng `5·10^7` bước — chậm). Bộ nhớ `O(n)`. Ưu điểm: **dễ hiểu, không đệ quy**; nhược điểm: chậm ở trường hợp xấu.

### Cách 4 — Tiền tự + **stack** (không đệ quy, `O(n)`)

**Ý tưởng.** Vẫn tiền tự, nhưng bỏ đệ quy bằng cách giữ một stack các nút "còn có thể nhận con phải" (đường đi từ gốc xuống nút vừa tạo). Với giá trị mới `x`:

- Nếu `x < đỉnh stack.val`: `x` là **con trái** của đỉnh (vì tiền tự chạy xuống con trái trước).
- Ngược lại: `x` phải là con phải của **một tổ tiên nào đó**. Pop khỏi stack mọi nút có `val < x`; nút bị pop **cuối cùng** là cha (tổ tiên nhỏ nhất mà `x` vẫn lớn hơn) và `x` là con phải của nó.
- Đẩy `x` vào stack.

Tương tự khi `serialize`: dùng stack thay đệ quy (như Cách 3). Cách này **chống tràn đệ quy** với cây lệch 10^4 nút.

In [13]:
class CodecStack:
    """Cách 4: tiền tự + stack, không đệ quy ở cả hai chiều. Đây là bản nộp an toàn nhất."""

    def serialize(self, root):
        out, stack = [], [root] if root else []
        while stack:
            node = stack.pop()
            out.append(node.val)
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
        return " ".join(map(str, out))

    def deserialize(self, data):
        nums = list(map(int, data.split()))
        if not nums:
            return None
        root = TreeNode(nums[0])
        stack = [root]
        for x in nums[1:]:
            node = TreeNode(x)
            if x < stack[-1].val:
                stack[-1].left = node
            else:
                parent = None
                while stack and stack[-1].val < x:
                    parent = stack.pop()
                parent.right = node
            stack.append(node)
        return root


for vals in ([2, 1, 3], [], [5, 3, 6, 2, 4, None, 7]):
    data, back = run_codec(CodecStack, build_tree(vals))
    assert same_tree(build_tree(vals), back)
    print(vals, "->", repr(data))

[2, 1, 3] -> '2 1 3'
[] -> ''
[5, 3, 6, 2, 4, None, 7] -> '5 3 2 4 6 7'


**Độ phức tạp.** Mỗi giá trị được `push` đúng một lần và `pop` tối đa một lần → thời gian `O(n)` (amortized), bộ nhớ `O(n)` (stack tối đa `h` nút). **Không đệ quy → không lo `RecursionError`.**

### Cách 5 — Nén gọn thật sự: mỗi số là **một ký tự** (`chr` / `ord`)

**Ý tưởng.** "Càng gọn càng tốt" — dạng chữ số `"1234 "` tốn tới 5 ký tự cho giá trị 1234. Vì `0 <= val <= 10^4 < 55296` (vùng ký tự Unicode hợp lệ không dính khoảng "surrogate"), ta ánh xạ **mỗi giá trị sang đúng một ký tự**: `chr(val)`. Chuỗi có đúng **`n` ký tự**, **không cần dấu cách**, vì mỗi số chiếm đúng một ký tự. Giải mã: `ord(c)` cho từng ký tự rồi dùng lại stack của Cách 4.

Hãy xem `chr`/`ord` trước.

In [14]:
print(chr(65), ord("A"))           # A 65
print(repr(chr(10000)), ord(chr(10000)))
s = "".join(chr(v) for v in [5, 3, 2, 4, 6, 7])
print(len(s), [ord(c) for c in s])     # 6 [5, 3, 2, 4, 6, 7]

A 65
'✐' 10000
6 [5, 3, 2, 4, 6, 7]


In [15]:
class CodecChr:
    """Cách 5: mỗi giá trị -> một ký tự chr(val); giải mã bằng stack."""

    def serialize(self, root):
        out, stack = [], [root] if root else []
        while stack:
            node = stack.pop()
            out.append(chr(node.val))
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
        return "".join(out)

    def deserialize(self, data):
        if not data:
            return None
        nums = [ord(c) for c in data]
        root = TreeNode(nums[0])
        stack = [root]
        for x in nums[1:]:
            node = TreeNode(x)
            if x < stack[-1].val:
                stack[-1].left = node
            else:
                parent = None
                while stack and stack[-1].val < x:
                    parent = stack.pop()
                parent.right = node
            stack.append(node)
        return root


big = build_tree([5000, 1234, 9999, 12, 4321, None, 10000])
for cls in (CodecNull, CodecBounds, CodecChr):
    data, back = run_codec(cls, big)
    assert same_tree(big, back)
    print(f"{cls.__name__:12s} dài {len(data):3d} ký tự, {len(data.encode('utf-8')):3d} byte UTF-8")

CodecNull    dài  42 ký tự,  42 byte UTF-8
CodecBounds  dài  28 ký tự,  28 byte UTF-8
CodecChr     dài   6 ký tự,  15 byte UTF-8


Lưu ý: so theo **số ký tự**, Cách 5 chỉ còn `n`; so theo **số byte UTF-8** thì ký tự lớn tốn 2–3 byte nên lợi thế nhỏ hơn. Thực tế người chấm LeetCode không đo độ gọn tuyệt đối, nhưng đây là hướng "gọn nhất" thường nhắc tới. Thử nén bằng cách ghi hai byte cố định cũng được, nhưng ta dừng ở đây.

**Độ phức tạp.** Giống Cách 4: thời gian `O(n)`, bộ nhớ `O(n)`; chuỗi ngắn nhất (`n` ký tự).

## 8. TRACE từng bước

Ta trace **Cách 2** (đệ quy khoảng) và **Cách 4** (stack) khi giải mã chuỗi `"5 3 2 4 6 7"` (cây `[5,3,6,2,4,null,7]`).

In [16]:
def trace_bounds(data):
    nums = list(map(int, data.split()))
    i = 0
    inf = float("inf")

    def dfs(lo, hi, depth, side):
        nonlocal i
        pad = "  " * depth
        if i == len(nums):
            print(f"{pad}{side} khoảng ({lo}, {hi}): hết dãy -> None")
            return None
        x = nums[i]
        if not (lo < x < hi):
            print(f"{pad}{side} khoảng ({lo}, {hi}): nums[{i}]={x} ngoài khoảng -> None (không tiêu thụ)")
            return None
        print(f"{pad}{side} khoảng ({lo}, {hi}): nums[{i}]={x} vừa -> tạo nút {x}")
        i += 1
        node = TreeNode(x)
        node.left = dfs(lo, x, depth + 1, "L")
        node.right = dfs(x, hi, depth + 1, "R")
        return node

    return dfs(-inf, inf, 0, "gốc")


root = trace_bounds("5 3 2 4 6 7")
print(tree_to_list(root))
assert tree_to_list(root) == [5, 3, 6, 2, 4, None, 7]

gốc khoảng (-inf, inf): nums[0]=5 vừa -> tạo nút 5
  L khoảng (-inf, 5): nums[1]=3 vừa -> tạo nút 3
    L khoảng (-inf, 3): nums[2]=2 vừa -> tạo nút 2
      L khoảng (-inf, 2): nums[3]=4 ngoài khoảng -> None (không tiêu thụ)
      R khoảng (2, 3): nums[3]=4 ngoài khoảng -> None (không tiêu thụ)
    R khoảng (3, 5): nums[3]=4 vừa -> tạo nút 4
      L khoảng (3, 4): nums[4]=6 ngoài khoảng -> None (không tiêu thụ)
      R khoảng (4, 5): nums[4]=6 ngoài khoảng -> None (không tiêu thụ)
  R khoảng (5, inf): nums[4]=6 vừa -> tạo nút 6
    L khoảng (5, 6): nums[5]=7 ngoài khoảng -> None (không tiêu thụ)
    R khoảng (6, inf): nums[5]=7 vừa -> tạo nút 7
      L khoảng (6, 7): hết dãy -> None
      R khoảng (7, inf): hết dãy -> None
[5, 3, 6, 2, 4, None, 7]


In [17]:
def trace_stack(data):
    nums = list(map(int, data.split()))
    root = TreeNode(nums[0])
    stack = [root]
    print(f"đọc {nums[0]}: gốc; stack = {[n.val for n in stack]}")
    for x in nums[1:]:
        node = TreeNode(x)
        if x < stack[-1].val:
            print(f"đọc {x}: {x} < đỉnh {stack[-1].val} -> con TRÁI của {stack[-1].val}", end="")
            stack[-1].left = node
        else:
            popped = []
            parent = None
            while stack and stack[-1].val < x:
                parent = stack.pop()
                popped.append(parent.val)
            print(f"đọc {x}: pop {popped} -> con PHẢI của {parent.val}", end="")
            parent.right = node
        stack.append(node)
        print(f";  stack = {[n.val for n in stack]}")
    return root


root = trace_stack("5 3 2 4 6 7")
assert tree_to_list(root) == [5, 3, 6, 2, 4, None, 7]

đọc 5: gốc; stack = [5]
đọc 3: 3 < đỉnh 5 -> con TRÁI của 5;  stack = [5, 3]
đọc 2: 2 < đỉnh 3 -> con TRÁI của 3;  stack = [5, 3, 2]
đọc 4: pop [2, 3] -> con PHẢI của 3;  stack = [5, 4]
đọc 6: pop [4, 5] -> con PHẢI của 5;  stack = [6]
đọc 7: pop [6] -> con PHẢI của 6;  stack = [7]


Bảng trace Cách 4 trên `"5 3 2 4 6 7"`:

| Đọc `x` | So với đỉnh stack | Hành động | Stack sau bước |
|---|---|---|---|
| 5 | — | gốc | `[5]` |
| 3 | 3 < 5 | `5.left = 3` | `[5, 3]` |
| 2 | 2 < 3 | `3.left = 2` | `[5, 3, 2]` |
| 4 | 4 > 2 | pop 2 (2 < 4); đỉnh 3 < 4 pop 3; đỉnh 5 > 4 dừng → cha = 3; `3.right = 4` | `[5, 4]` |
| 6 | 6 > 4 | pop 4, pop 5 → cha = 5; `5.right = 6` | `[6]` |
| 7 | 7 > 6 | pop 6 → cha = 6; `6.right = 7` | `[7]` |

Ý nghĩa: stack luôn là **đường đi từ gốc xuống nút gần nhất** nhưng sau khi pop chỉ giữ những nút có thể còn nhận con phải; "cha của `x`" là nút bị pop cuối cùng.

## 9. Code nộp LeetCode

Bản nộp chọn **Cách 4** (tiền tự + stack, hoàn toàn không đệ quy): `O(n)` thời gian, an toàn với cây lệch 10^4 nút. Giữ đúng tên lớp `Codec`, hai phương thức `serialize`, `deserialize` và **không lưu trạng thái trong đối tượng**.

In [18]:
class Codec:
    def serialize(self, root: Optional[TreeNode]) -> str:
        """Encodes a tree to a single string."""
        out, stack = [], [root] if root else []
        while stack:
            node = stack.pop()
            out.append(node.val)
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
        return " ".join(map(str, out))

    def deserialize(self, data: str) -> Optional[TreeNode]:
        """Decodes your encoded data to tree."""
        nums = list(map(int, data.split()))
        if not nums:
            return None
        root = TreeNode(nums[0])
        stack = [root]
        for x in nums[1:]:
            node = TreeNode(x)
            if x < stack[-1].val:
                stack[-1].left = node
            else:
                parent = None
                while stack and stack[-1].val < x:
                    parent = stack.pop()
                parent.right = node
            stack.append(node)
        return root


# Your Codec object will be instantiated and called as such:
# ser = Codec()
# deser = Codec()
# tree = ser.serialize(root)
# ans = deser.deserialize(tree)
# return ans
for vals in ([2, 1, 3], [], [5, 3, 6, 2, 4, None, 7]):
    data, back = run_codec(Codec, build_tree(vals))
    assert tree_to_list(back) == vals
    print(vals, "->", repr(data), "->", tree_to_list(back))

[2, 1, 3] -> '2 1 3' -> [2, 1, 3]
[] -> '' -> []
[5, 3, 6, 2, 4, None, 7] -> '5 3 2 4 6 7' -> [5, 3, 6, 2, 4, None, 7]


## 10. Vì sao thuật toán đúng

**Khẳng định.** Với BST (giá trị đôi một khác nhau), dãy tiền tự xác định duy nhất cây.

**Chứng minh (quy nạp theo số nút).** Dãy rỗng ↔ cây rỗng. Với cây `T` có gốc `r`, con trái `L`, con phải `R`: tiền tự là `r, pre(L), pre(R)`. Do tính chất BST, mọi giá trị của `L` `< r` và mọi giá trị của `R` `> r`. Vậy từ dãy tiền tự, phần tử đầu là `r`, phần còn lại gồm **một đoạn liền các số `< r`** (chính là `pre(L)`) rồi **một đoạn liền các số `> r`** (chính là `pre(R)`). Hai đoạn này xác định `L` và `R` duy nhất theo giả thiết quy nạp. ∎

**Vì sao cách dùng khoảng `(lo, hi)` đúng.** Khi dựng cây con có khoảng `(lo, hi)` và giá trị kế tiếp nằm trong khoảng, nó là gốc (vì tiền tự đặt gốc trước toàn bộ cây con); nếu nằm ngoài khoảng thì cây con rỗng vì mọi giá trị còn lại **của cây con này** đã được tiêu thụ hết (tiền tự đặt chúng liền nhau ngay sau gốc). **Vì sao stack đúng:** là bản khử đệ quy của chính điều đó; phần tử `x` lớn hơn đỉnh thì phải ở nhánh phải của tổ tiên gần nhất nhỏ hơn `x`.

Kiểm chứng tính chất "nhóm nhỏ hơn gốc đứng liền sau gốc" trên nhiều cây ngẫu nhiên:

In [19]:
def random_bst(rng, n, lo=0, hi=10**4):
    """BST ngẫu nhiên n nút, giá trị khác nhau, chèn theo thứ tự ngẫu nhiên."""
    vals = rng.sample(range(lo, hi + 1), n)
    root = None
    for v in vals:
        if root is None:
            root = TreeNode(v)
            continue
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(v); break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(v); break
                cur = cur.right
    return root


rng = random.Random(449)
for _ in range(300):
    t = random_bst(rng, rng.randint(1, 30))
    pre = preorder(t)
    r = pre[0]
    k = sum(1 for x in pre[1:] if x < r)
    assert all(x < r for x in pre[1:1 + k]) and all(x > r for x in pre[1 + k:])
print("Tính chất 'tiền tự = gốc, đoạn nhỏ hơn, đoạn lớn hơn' đúng trên 300 cây ngẫu nhiên.")

Tính chất 'tiền tự = gốc, đoạn nhỏ hơn, đoạn lớn hơn' đúng trên 300 cây ngẫu nhiên.


## 11. Phân tích Time / Space Complexity

Với `n` là số nút, `h` là chiều cao cây (`log n ≤ h ≤ n`):

| Cách | `serialize` | `deserialize` | Space (ngoài kết quả) | Độ dài chuỗi |
|---|---|---|---|---|
| 1. Có `#` | `O(n)` | `O(n)` | `O(h)` | ≈ `2n + 1` token |
| 2. Tiền tự + `(lo, hi)` | `O(n)` | `O(n)` | `O(h)` đệ quy | `n` số + dấu cách |
| 3. Chèn lần lượt | `O(n)` | `O(n·h)` (xấu nhất `O(n²)`) | `O(1)` thêm | như Cách 2 |
| 4. Stack | `O(n)` | `O(n)` | `O(h)` stack tường minh | như Cách 2 |
| 5. `chr` | `O(n)` | `O(n)` | `O(h)` | **`n` ký tự** |

Bài này luôn phải đọc mỗi nút ít nhất một lần nên `O(n)` là tối ưu. Khác biệt chính nằm ở **độ gọn của chuỗi** và **an toàn đệ quy**.

## 12. Bộ test

### 12.1. Ví dụ của đề và các cách đối chiếu nhau

In [20]:
ALL_CODECS = [CodecNull, CodecBounds, CodecInsert, CodecStack, CodecChr, Codec]


def check_codec(cls, root):
    data, back = run_codec(cls, root)       # ser và deser là hai đối tượng khác nhau
    assert same_tree(root, back), (cls.__name__, tree_to_list(root), repr(data))
    return data


for vals in ([2, 1, 3], []):
    for cls in ALL_CODECS:
        check_codec(cls, build_tree(vals))
print("Hai ví dụ của đề: tất cả các cách đều khứ hồi đúng.")

Hai ví dụ của đề: tất cả các cách đều khứ hồi đúng.


### 12.2. Edge case

Cây rỗng, một nút, giá trị `0`, giá trị lớn nhất `10^4`, chuỗi lệch trái, chuỗi lệch phải, cây zigzag.

In [21]:
def chain_left(n):      # n, n-1, ..., 1 (mỗi nút chỉ có con trái)
    root = None
    for v in range(1, n + 1):
        root = TreeNode(v, left=root)
    return root


def chain_right(n):     # 1, 2, ..., n (mỗi nút chỉ có con phải)
    root = None
    for v in range(n, 0, -1):
        root = TreeNode(v, right=root)
    return root


edge_trees = {
    "rỗng": None,
    "một nút 0": TreeNode(0),
    "một nút 10000": TreeNode(10000),
    "hai nút 0,10000": build_tree([0, None, 10000]),
    "hai nút 10000,0": build_tree([10000, 0]),
    "lệch trái 5": chain_left(5),
    "lệch phải 5": chain_right(5),
    "zigzag": build_tree([5, 2, None, None, 4, 3]),
}
for name, t in edge_trees.items():
    for cls in ALL_CODECS:
        check_codec(cls, t)
    print(f"{name:18s} OK, chuỗi Cách 4 =", repr(Codec().serialize(t)))

rỗng               OK, chuỗi Cách 4 = ''
một nút 0          OK, chuỗi Cách 4 = '0'
một nút 10000      OK, chuỗi Cách 4 = '10000'
hai nút 0,10000    OK, chuỗi Cách 4 = '0 10000'
hai nút 10000,0    OK, chuỗi Cách 4 = '10000 0'
lệch trái 5        OK, chuỗi Cách 4 = '5 4 3 2 1'
lệch phải 5        OK, chuỗi Cách 4 = '1 2 3 4 5'
zigzag             OK, chuỗi Cách 4 = '5 2 4 3'


### 12.3. Phản ví dụ cho các cách làm sai

**(a) Chỉ ghi trung tự (inorder) là KHÔNG đủ.** Nhiều BST khác nhau có cùng dãy trung tự (luôn là dãy sắp xếp) nên không khôi phục được hình dạng.

**(b) Giải mã "ngây thơ" luôn gắn vào con trái** sẽ sai hình dạng.

In [22]:
t1 = build_tree([2, 1, 3])               # cân bằng
t2 = build_tree([1, None, 2, None, 3])   # chuỗi phải
print("inorder t1:", inorder(t1), "| inorder t2:", inorder(t2))
# Hai cây khác nhau nhưng (nếu chỉ lưu inorder của cùng tập giá trị 1,2,3) chuỗi giống hệt
assert inorder(t1) == inorder(t2) and not same_tree(t1, t2)

# Phản ví dụ (b): giải mã sai - luôn gắn nút kế tiếp vào con trái
def wrong_deserialize(data):
    nums = list(map(int, data.split()))
    root = cur = None
    for x in nums:
        node = TreeNode(x)
        if root is None:
            root = cur = node
        else:
            cur.left = node
            cur = node
    return root


data = Codec().serialize(build_tree([2, 1, 3]))      # '2 1 3'
print("giải mã sai cho:", tree_to_list(wrong_deserialize(data)), "(đáng lẽ [2, 1, 3])")
assert not same_tree(wrong_deserialize(data), build_tree([2, 1, 3]))

inorder t1: [1, 2, 3] | inorder t2: [1, 2, 3]
giải mã sai cho: [2, 1, None, 3] (đáng lẽ [2, 1, 3])


### 12.4. Đối chiếu ngẫu nhiên (có seed) giữa các cách

Với hàng trăm BST ngẫu nhiên đúng ràng buộc (giá trị trong `[0, 10^4]`, `n` đến vài trăm), mọi cách phải khứ hồi đúng; các cách "text" phải cho **cùng một chuỗi**; Cách 5 phải ngắn đúng `n` ký tự.

In [23]:
rng = random.Random(20240449)
for trial in range(400):
    n = rng.randint(0, 200)
    t = random_bst(rng, n) if n else None
    texts = {}
    for cls in ALL_CODECS:
        texts[cls.__name__] = check_codec(cls, t)
    assert texts["CodecBounds"] == texts["CodecInsert"] == texts["CodecStack"] == texts["Codec"]
    assert len(texts["CodecChr"]) == n
    # chuỗi có # dài đúng 2n+1 token
    assert len(texts["CodecNull"].split()) == 2 * n + 1
print("400 BST ngẫu nhiên: mọi cách đều khứ hồi đúng, cùng chuỗi tiền tự, độ dài khớp lý thuyết.")

400 BST ngẫu nhiên: mọi cách đều khứ hồi đúng, cùng chuỗi tiền tự, độ dài khớp lý thuyết.


### 12.5. Cây lệch lớn 10^4 nút và hiệu năng

Cây lệch tối đa theo ràng buộc: `10^4` nút. Cách đệ quy (Cách 1, 2) có thể gặp `RecursionError` với giới hạn đệ quy mặc định (`1000`) của Python; các cách dùng stack thì không.

In [24]:
N = 10**4
deep = chain_right(N)
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

for cls in (CodecBounds, CodecNull):
    try:
        check_codec(cls, deep)
        print(f"{cls.__name__:12s}: chạy được")
    except RecursionError:
        print(f"{cls.__name__:12s}: RecursionError (đệ quy quá sâu)")

for cls in (CodecStack, CodecChr, Codec):
    t0 = time.perf_counter()
    check_codec(cls, deep)
    check_codec(cls, chain_left(N))
    print(f"{cls.__name__:12s}: OK trên 2 chuỗi 10^4 nút, {time.perf_counter() - t0:.3f}s")

giới hạn đệ quy hiện tại: 1000
CodecBounds : RecursionError (đệ quy quá sâu)
CodecNull   : RecursionError (đệ quy quá sâu)
CodecStack  : OK trên 2 chuỗi 10^4 nút, 0.018s
CodecChr    : OK trên 2 chuỗi 10^4 nút, 0.013s
Codec       : OK trên 2 chuỗi 10^4 nút, 0.013s


In [25]:
# Cách 3 (chèn lần lượt) chậm ở trường hợp xấu: so sánh cây ngẫu nhiên và cây lệch, cùng n = 3000
for name, t in [("ngẫu nhiên", random_bst(random.Random(1), 3000)), ("chuỗi lệch", chain_right(3000))]:
    data = CodecStack().serialize(t)
    t0 = time.perf_counter()
    CodecInsert().deserialize(data)
    t_insert = time.perf_counter() - t0
    t0 = time.perf_counter()
    CodecStack().deserialize(data)
    t_stack = time.perf_counter() - t0
    print(f"{name:11s}: chèn lần lượt {t_insert:.4f}s | stack {t_stack:.4f}s")

ngẫu nhiên : chèn lần lượt 0.0021s | stack 0.0010s
chuỗi lệch : chèn lần lượt 0.1266s | stack 0.0009s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Lưu trạng thái trong đối tượng `Codec`

LeetCode dùng **hai đối tượng khác nhau** cho `serialize` và `deserialize`, nên thông tin lưu trong `self` ở bước 1 **không còn** ở bước 2.

In [26]:
class BadCodecState:
    def serialize(self, root):
        self.saved = root                  # SAI: cất cây vào self
        return "dummy"

    def deserialize(self, data):
        return self.saved                  # ser khác deser -> thuộc tính không tồn tại


try:
    run_codec(BadCodecState, build_tree([2, 1, 3]))
except AttributeError as e:
    print("Lỗi:", e)

Lỗi: 'BadCodecState' object has no attribute 'saved'


**Vì sao sai:** `deser` là đối tượng mới nên không có thuộc tính `saved`. **Cách đúng:** mọi thông tin phải nằm trong chuỗi trả về (mục 9).

### Lỗi 2: Quên cây rỗng / chuỗi rỗng

In [27]:
def bad_deserialize(data):
    nums = list(map(int, data.split()))
    root = TreeNode(nums[0])               # SAI: nums rỗng -> IndexError
    return root


try:
    bad_deserialize("")
except IndexError as e:
    print("Lỗi:", e)

# Đúng: kiểm tra rỗng trước
print(Codec().serialize(None) == "", Codec().deserialize("") is None)

Lỗi: list index out of range
True True


**Vì sao sai:** cây rỗng là test có thật (Ví dụ 2). `serialize(None)` nên trả `""` và `deserialize("")` nên trả `None`.

### Lỗi 3: Đổi thứ tự dựng con (phải trước trái) hoặc nhầm cận

Tiền tự là *nút → trái → phải*. Nếu giải mã dựng **phải trước** hoặc đặt cận `(x, hi)` cho con trái, cây sai hoặc chuỗi bị tiêu thụ sai chỗ.

In [28]:
def wrong_bounds_deserialize(data):
    nums = list(map(int, data.split()))
    i = 0
    inf = float("inf")

    def dfs(lo, hi):
        nonlocal i
        if i == len(nums) or not (lo < nums[i] < hi):
            return None
        x = nums[i]
        i += 1
        node = TreeNode(x)
        node.right = dfs(x, hi)            # SAI: dựng phải trước
        node.left = dfs(lo, x)
        return node

    return dfs(-inf, inf)


data = Codec().serialize(build_tree([5, 3, 6, 2, 4, None, 7]))
bad = wrong_bounds_deserialize(data)
print("serialize :", repr(data))
print("giải mã sai:", tree_to_list(bad), "(đáng lẽ [5, 3, 6, 2, 4, None, 7])")
assert not same_tree(bad, build_tree([5, 3, 6, 2, 4, None, 7]))

serialize : '5 3 2 4 6 7'
giải mã sai: [5, 3, None, 2] (đáng lẽ [5, 3, 6, 2, 4, None, 7])


**Vì sao sai:** tiền tự ghi trái trước nên giá trị kế tiếp (nếu nhỏ hơn gốc) thuộc **trái**; dựng phải trước thì dãy "nhỏ hơn" bị bỏ qua (vì ngoài khoảng `(x, hi)`), rồi mới tiêu thụ ở trái, nhưng lúc đó đã mất vị trí. **Đúng:** luôn `left = dfs(lo, x)` rồi `right = dfs(x, hi)`.

### Lỗi 4: Quên `nonlocal i`

In [29]:
def bad_nonlocal(data):
    nums = list(map(int, data.split()))
    i = 0

    def dfs(lo, hi):
        if i == len(nums) or not (lo < nums[i] < hi):
            return None
        x = nums[i]
        i += 1                              # SAI: gán i -> Python coi i là biến cục bộ của dfs
        return TreeNode(x)

    return dfs(float("-inf"), float("inf"))


try:
    bad_nonlocal("5 3 2")
except UnboundLocalError as e:
    print("Lỗi:", e)

Lỗi: cannot access local variable 'i' where it is not associated with a value


**Vì sao sai:** khi trong hàm có phép gán `i += 1`, Python xem `i` là biến cục bộ; đọc nó trước khi gán sinh `UnboundLocalError`. **Đúng:** khai báo `nonlocal i` ở đầu `dfs` (như Cách 2).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"BST" + "tuần tự hoá" → chỉ cần một thứ tự duyệt (tiền tự hoặc hậu tự)**; trung tự một mình thì không đủ. Với cây nhị phân thường mới cần `#`.
- **Tiền tự + khoảng `(lo, hi)`** là mẫu chung "dựng BST từ tiền tự" (cũng là bài *Construct BST from Preorder Traversal*, 1008). Hậu tự dùng đối xứng: đọc từ **cuối** dãy, dựng phải trước rồi trái.
- **Đệ quy sâu 10^4 trong Python** là rủi ro: nếu chưa chắc LeetCode nâng giới hạn, dùng **stack tường minh** hoặc `sys.setrecursionlimit`.
- **Bài design**: kiểm tra mọi thông tin đều nằm trong chuỗi; viết bộ chạy `run_codec` với hai đối tượng riêng như mục 3.6.
- **Độ gọn**: loại bỏ mọi thứ suy ra được. Ở đây `#`, dấu ngoặc, độ sâu đều suy ra được từ tính chất BST.
- Luôn thử **cây rỗng**, **một nút**, **chuỗi lệch hai phía**.

## 15. Biến thể của bài

1. **BST có giá trị trùng.** Tiền tự trở nên mơ hồ (giá trị trùng có thể là con trái hoặc con phải). Cần quy ước (ví dụ: trùng luôn ở bên phải) thì khoảng thành `[x, hi)` thay vì `(x, hi)`; hoặc ghi thêm thông tin.
2. **Cây nhị phân bất kỳ** (bài 297): phải ghi `#` như Cách 1 (hoặc ghi tiền tự + trung tự) vì không có tính chất thứ tự.
3. **Hậu tự thay tiền tự**: đọc dãy hậu tự từ cuối, `dfs(lo, hi)` dựng **phải trước, trái sau**.
4. **Cây N-ary** (bài 428/431): ghi số con kèm giá trị, hoặc dùng ký hiệu đóng ngoặc.
5. **Nén bit**: mỗi giá trị `≤ 10^4` cần `14` bit; ghép các giá trị thành chuỗi bit rồi mã hoá base64 để dùng ký tự in được.

Ví dụ nhỏ cho biến thể 3 (hậu tự):

In [30]:
def serialize_post(root):
    out = []

    def dfs(node):
        if node is None:
            return
        dfs(node.left)
        dfs(node.right)
        out.append(node.val)

    dfs(root)
    return " ".join(map(str, out))


def deserialize_post(data):
    nums = list(map(int, data.split()))
    inf = float("inf")

    def dfs(lo, hi):
        if not nums or not (lo < nums[-1] < hi):
            return None
        x = nums.pop()                 # lấy từ CUỐI: nút cha đứng sau con trong hậu tự
        node = TreeNode(x)
        node.right = dfs(x, hi)        # phải TRƯỚC, rồi mới tới trái
        node.left = dfs(lo, x)
        return node

    return dfs(-inf, inf)


t = build_tree([5, 3, 6, 2, 4, None, 7])
s = serialize_post(t)
print(repr(s), tree_to_list(deserialize_post(s)))
assert same_tree(t, deserialize_post(s))

'2 4 3 7 6 5' [5, 3, 6, 2, 4, None, 7]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Có `#` | tiền tự tổng quát, ghi nút rỗng | `O(n)` | `O(h)` đệ quy | cây nhị phân bất kỳ; không tận dụng BST |
| 2. `(lo, hi)` | tiền tự + khoảng cho phép | `O(n)` | `O(h)` đệ quy | cách chuẩn khi cây không quá lệch |
| 3. Chèn lần lượt | chèn theo tiền tự vào BST | `O(n·h)` | `O(1)` thêm | dễ nghĩ, ít code; chậm với cây lệch |
| 4. Stack | khử đệ quy bằng stack | `O(n)` | `O(h)` | **nộp bài an toàn**, cây sâu |
| 5. `chr` | mỗi giá trị một ký tự | `O(n)` | `O(h)` | khi cần chuỗi **gọn nhất** |

Quy tắc chọn: muốn **an toàn + nhanh** → Cách 4; muốn **nhiều điểm về độ gọn** → Cách 5; học **ý nghĩa khoảng** → Cách 2.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Codec().serialize(build_tree([4, 2, 6]))`, `Codec().serialize(None)`, `tree_to_list(Codec().deserialize("2 1 3"))`, `tree_to_list(Codec().deserialize("1 2 3"))`, `tree_to_list(Codec().deserialize("3 2 1"))`.

**Bài 2 (tính tay).** Từ chuỗi tiền tự `"8 3 1 6 4 7 10 14 13"`, vẽ cây và viết dạng list kiểu LeetCode. Sau đó viết dãy hậu tự của cây đó.

**Bài 3 (điền TODO).** Hoàn thành `deserialize_todo(data)` theo Cách 2 (khoảng `(lo, hi)`).

In [31]:
def deserialize_todo(data):
    # TODO 1: nums = list(map(int, data.split())); i = 0
    # TODO 2: viết dfs(lo, hi) với nonlocal i:
    #         nếu i == len(nums) hoặc nums[i] ngoài (lo, hi) -> trả None
    #         x = nums[i]; i += 1; node = TreeNode(x)
    #         node.left = dfs(lo, x); node.right = dfs(x, hi); trả node
    # TODO 3: return dfs(-inf, inf)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết lớp `CodecLevel` dùng **duyệt theo tầng (BFS)** để `serialize` (không có `#`): ghi các giá trị theo thứ tự tầng. Giải mã bằng cách **chèn lần lượt** (như Cách 3). Giải thích vì sao chèn theo thứ tự tầng cũng khôi phục đúng hình dạng cây.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_nodes_in_string(data)` trả về số nút **mà không dựng cây** (đếm token). (b) Viết `serialize_range(root, lo, hi)` chỉ ghi các giá trị nằm trong `[lo, hi]` theo tiền tự — và chứng minh bằng `assert` rằng khi giải mã bằng Cách 4 ta được **một BST hợp lệ** chứa đúng các giá trị đó (không nhất thiết cùng hình dạng với cây lọc).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `'4 2 6'`, `''`, `[2, 1, 3]`, `[1, None, 2, None, 3]` (chuỗi phải), `[3, 2, None, 1]` (chuỗi trái). Bài 3: bám sát mã Cách 2 ở mục 7.
- **Mức 2 (vừa):** Bài 2: gốc `8`; nhóm nhỏ hơn 8 là `3 1 6 4 7`, nhóm lớn hơn là `10 14 13`; lặp lại cho từng nhóm. Bài 4: BFS cho ra mỗi cha đứng trước các con của nó; khi chèn, cha luôn đã có sẵn nên đường đi dẫn đúng chỗ.
- **Mức 3 (sâu):** Bài 5(a): số token = số nút (không có `#`). 5(b): lọc theo tiền tự cho dãy vẫn là tiền tự của **một BST** (cây lọc không cần giữ hình dạng, nhưng nếu chèn lần lượt các giá trị lọc theo thứ tự đó thì được BST hợp lệ); dùng hàm kiểm tra inorder tăng dần.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert Codec().serialize(build_tree([4, 2, 6])) == "4 2 6"
assert Codec().serialize(None) == ""
assert tree_to_list(Codec().deserialize("2 1 3")) == [2, 1, 3]
assert tree_to_list(Codec().deserialize("1 2 3")) == [1, None, 2, None, 3]
assert tree_to_list(Codec().deserialize("3 2 1")) == [3, 2, None, 1]

# Bài 2: "8 3 1 6 4 7 10 14 13"
t2 = Codec().deserialize("8 3 1 6 4 7 10 14 13")
assert tree_to_list(t2) == [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]
print("Bài 2 - hậu tự:", serialize_post(t2))
assert serialize_post(t2) == "1 4 7 6 3 13 14 10 8"


# Bài 3
def deserialize_todo_solution(data):
    nums = list(map(int, data.split()))
    i = 0
    inf = float("inf")

    def dfs(lo, hi):
        nonlocal i
        if i == len(nums) or not (lo < nums[i] < hi):
            return None
        x = nums[i]
        i += 1
        node = TreeNode(x)
        node.left = dfs(lo, x)
        node.right = dfs(x, hi)
        return node

    return dfs(-inf, inf)


for s in ["", "2 1 3", "8 3 1 6 4 7 10 14 13"]:
    assert same_tree(deserialize_todo_solution(s), Codec().deserialize(s))


# Bài 4
class CodecLevel:
    def serialize(self, root):
        out, queue = [], deque([root] if root else [])
        while queue:
            node = queue.popleft()
            out.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        return " ".join(map(str, out))

    def deserialize(self, data):
        return CodecInsert().deserialize(data)       # chèn lần lượt theo thứ tự tầng


rng = random.Random(7)
for _ in range(200):
    n = rng.randint(0, 60)
    t = random_bst(rng, n) if n else None
    check_codec(CodecLevel, t)


# Bài 5
def count_nodes_in_string(data):
    return len(data.split())


def is_bst(root):
    vals = []
    stack, cur = [], root
    while stack or cur:
        while cur:
            stack.append(cur); cur = cur.left
        cur = stack.pop(); vals.append(cur.val); cur = cur.right
    return all(a < b for a, b in zip(vals, vals[1:])), vals


def serialize_range(root, lo, hi):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        if lo <= node.val <= hi:
            out.append(node.val)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return " ".join(map(str, out))


rng = random.Random(99)
for _ in range(200):
    n = rng.randint(0, 50)
    t = random_bst(rng, n) if n else None
    assert count_nodes_in_string(Codec().serialize(t)) == n
    lo = rng.randint(0, 5000)
    hi = lo + rng.randint(0, 5000)
    s = serialize_range(t, lo, hi)
    rebuilt = CodecInsert().deserialize(s)
    ok, vals = is_bst(rebuilt)
    assert ok and vals == sorted(v for v in inorder(t) if lo <= v <= hi)
print("Tất cả đáp án bài tập đều đúng.")

Bài 2 - hậu tự: 1 4 7 6 3 13 14 10 8
Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (BST → chuỗi gọn bằng tiền tự + stack)

```python
class Codec:
    def serialize(self, root):
        out, stack = [], [root] if root else []
        while stack:
            node = stack.pop()
            out.append(node.val)
            if node.right: stack.append(node.right)   # phải trước để trái ra trước
            if node.left:  stack.append(node.left)
        return " ".join(map(str, out))

    def deserialize(self, data):
        nums = list(map(int, data.split()))
        if not nums:
            return None
        root = TreeNode(nums[0]); stack = [root]
        for x in nums[1:]:
            node = TreeNode(x)
            if x < stack[-1].val:
                stack[-1].left = node                 # nhỏ hơn đỉnh: con trái
            else:
                parent = None
                while stack and stack[-1].val < x:    # lớn hơn: tìm tổ tiên nhỏ nhất mà x vẫn lớn hơn
                    parent = stack.pop()
                parent.right = node
            stack.append(node)
        return root
```

**Checklist**

- [ ] Chuỗi chỉ cần **một** thứ tự duyệt (tiền tự) vì là BST; không ghi `#`?
- [ ] Cây rỗng: `serialize(None) == ""` và `deserialize("") is None`?
- [ ] Không lưu trạng thái trong `self` (LeetCode dùng hai đối tượng khác nhau)?
- [ ] Dựng con **trái trước, phải sau**; cận `(lo, x)` cho trái, `(x, hi)` cho phải?
- [ ] Có lo `RecursionError` với cây lệch 10^4 nút? (đã dùng stack tường minh chưa?)
- [ ] Đã thử: `[]`, một nút, `0`, `10000`, chuỗi trái, chuỗi phải, cây ngẫu nhiên khứ hồi `same_tree`?
- [ ] Skeleton đúng `class Codec` / `serialize(self, root) -> str` / `deserialize(self, data) -> Optional[TreeNode]`?